# Week 5 build · A character-level GPT on tiny Shakespeare

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mjtri/learnml/blob/main/notebooks/week-05.ipynb)

**~3 hours · T4 for Part D (Runtime → Change runtime type → T4) · run top to bottom.**

**The one rule:** every experiment is preceded by a 🔮 **predict cell**. Type what you expect and how sure you are *before* running the experiment. The notebook refuses empty predictions. A rough guess is fine; the gap between guess and result is what you are here to collect (canvas steps 7–8: *predict before running*, *log the surprise*).

**Two passes.** First run with `SMOKE = True` (next cell): a 20 KB slice, a 2-block model, ~150 steps, under two minutes on CPU, so you see every cell work. Then set `SMOKE = False`, switch to the T4, and run again for real (~15 min of training in Part D, ~25 more in Part F).

| Part | Topic | Time |
|---|---|---|
| A | Data: tiny Shakespeare, the alphabet, (input, target) batches | 20 min |
| B | Attention and the transformer block; what survives eight blocks with and without the residual add | 25 min |
| C | Assemble the GPT; predict the parameter count, then check it | 30 min |
| D | Train, read the loss curve, save a checkpoint, log the run | 50 min |
| E | Sampling: greedy, temperature, top-k, and reproducing a sample from its seed | 25 min |
| F | Figure 1 → code map; stretch: pre-norm vs post-norm, loss per depth | 30 min |


In [ ]:
#@title Setup: run me first
import numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Audio, Markdown, display
torch.manual_seed(0); np.random.seed(0)
_LEDGER = {}

def predict(tag, prediction, confidence):
    """Log a prediction BEFORE the experiment. Refuses empty predictions."""
    if not str(prediction).strip():
        raise ValueError(f"[{tag}] Write a prediction first. A rough guess is fine; an empty one is not.")
    _LEDGER[tag] = {"prediction": str(prediction).strip(), "confidence": int(confidence), "actual": "", "surprise": ""}
    print(f"Logged [{tag}] at {confidence}% confidence: {prediction}")

def record(tag, what_happened, surprise="none"):
    if tag not in _LEDGER:
        raise ValueError(f"[{tag}] has no prediction yet. Run its predict cell first.")
    _LEDGER[tag].update(actual=str(what_happened).strip(), surprise=surprise)
    print(f"Recorded [{tag}] surprise={surprise}")

def reveal():
    rows = ["| # | prediction | sure | what happened | surprise |", "|---|---|---|---|---|"]
    rows += [f"| {t} | {r['prediction']} | {r['confidence']}% | {r['actual']} | {r['surprise']} |" for t, r in _LEDGER.items()]
    print("\n".join(rows)); display(Markdown("\n".join(rows)))

print("torch", torch.__version__, "| ready")

---
## Part A · Data (20 min)

**Words used in this part.** *tiny Shakespeare*: one 1 MB text file of the plays. *character-level model*: every character is one token, so the alphabet is the whole "tokenizer". *block_size*: the context length, how many characters the model sees at once. *batch*: several context windows stacked along the first axis.

In [ ]:
#@title Config: SMOKE = True is a two-minute CPU rehearsal; False is the real T4 run
SMOKE = True  #@param {type:"boolean"}
import math, time, json, os, urllib.request, torch, torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
if SMOKE:
    cfg = dict(n_layer=2, n_embd=64, n_head=4, block_size=64, batch_size=32, steps=150, lr=1e-3, eval_every=50, seed=1337)
else:
    cfg = dict(n_layer=6, n_embd=384, n_head=6, block_size=256, batch_size=64, steps=3000, lr=3e-4, eval_every=250, seed=1337)
if not SMOKE and device == "cpu":
    print("WARNING: SMOKE=False on a CPU would take hours. Runtime -> Change runtime type -> T4.")
torch.manual_seed(cfg["seed"])
print("device:", device, "| config:", cfg)

In [ ]:
#@title 🔮 Predict A1: How many distinct characters (the alphabet size) will the text contain? Letters, both cases, punctuation, newline...
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("A1", prediction, confidence)

In [ ]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(URL, timeout=60).read().decode("utf-8")
if SMOKE:
    text = text[:20_000]                          # a 20 KB slice keeps the CPU rehearsal fast
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]           # text -> list of integers
decode = lambda l: "".join(itos[i] for i in l)    # list of integers -> text
print(f"{len(text):,} characters, alphabet of {vocab_size}:")
print(repr("".join(chars)))
print(text[:150])

Surprise? That alphabet *is* the whole vocabulary of this model: no words, no sub-word pieces. It has to learn spelling, then names, then line structure, in that order.

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]          # the last 10% is never trained on

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - cfg["block_size"], (cfg["batch_size"],))
    x = torch.stack([d[i:i + cfg["block_size"]] for i in ix])
    y = torch.stack([d[i + 1:i + cfg["block_size"] + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("x", tuple(xb.shape), "y", tuple(yb.shape))
print("input :", repr(decode(xb[0, :40].tolist())))
print("target:", repr(decode(yb[0, :40].tolist())))    # the same text shifted one character left

The target is the input shifted by one: at every position the model must guess the *next* character. One batch therefore contains `batch_size × block_size` separate prediction problems.

---
## Part B · The residual stream in numbers (25 min)

**Words used in this part.** *residual stream*: the running sum `x` that every block reads and adds to. *transformer block*: attention, then the MLP, each added back. *LayerNorm*: per token, subtract the mean and divide by the standard deviation. *causal mask*: a position may only look at itself and earlier positions.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.n_head = n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)        # query, key, value in one matmul
        self.c_proj = nn.Linear(n_embd, n_embd)            # output projection
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)))
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)   # (B, heads, T, head_size)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))          # scaled dot-product
        att = att.masked_fill(self.mask[:T, :T] == 0, float("-inf"))     # causal mask
        att = F.softmax(att, dim=-1)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)           # concatenate the heads
        return self.c_proj(y)

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.c_fc, self.gelu, self.c_proj = nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd)
    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))       # widen to 4d, bend, narrow back to d

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, norm="pre"):
        super().__init__()
        self.ln_1, self.ln_2, self.norm = nn.LayerNorm(n_embd), nn.LayerNorm(n_embd), norm
        self.attn, self.mlp = CausalSelfAttention(n_embd, n_head, block_size), MLP(n_embd)
    def forward(self, x):
        if self.norm == "pre":                     # GPT-2 / nanoGPT: the stream stays a clean sum
            x = x + self.attn(self.ln_1(x))
            x = x + self.mlp(self.ln_2(x))
        else:                                      # the 2017 figure: normalise the sum after each add
            x = self.ln_1(x + self.attn(x))
            x = self.ln_2(x + self.mlp(x))
        return x

In [ ]:
#@title 🔮 Predict B1: Push one random token vector through 8 UNTRAINED blocks. Cosine similarity between output and input: with the residual add, and without? (two numbers)
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("B1", prediction, confidence)

In [ ]:
torch.manual_seed(0)
d = cfg["n_embd"]
blocks = [Block(d, cfg["n_head"], cfg["block_size"]) for _ in range(8)]
x0 = torch.randn(1, 8, d)                                    # one sequence of 8 tokens, before any block
cos = lambda a, b: F.cosine_similarity(a.flatten(), b.flatten(), dim=0).item()
with torch.no_grad():
    x_res, x_plain = x0, x0
    print("block   cos(x, x0) with the add   cos(x, x0) without the add")
    for i, blk in enumerate(blocks):
        x_res = blk(x_res)                                                  # x + attn(ln(x)); x + mlp(ln(x))
        x_plain = blk.mlp(blk.ln_2(blk.attn(blk.ln_1(x_plain))))            # the same parts, no adds
        print(f"{i + 1:5d}   {cos(x_res, x0):10.3f}              {cos(x_plain, x0):10.3f}")

In [ ]:
#@title ✍️ B1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("B1", what_happened, surprise)

Without the add, the input's direction is gone within a block or two: each block *replaces* the stream. With it, the stream after eight blocks still points mostly the way the embedding did. The gradient uses the same highway in reverse.

---
## Part C · Assemble the GPT and count it (30 min)

**Words used in this part.** *parameter count*: total learned weights; about `12·d²` per block plus the embedding tables. *weight tying*: the output layer reuses the token-embedding matrix, so it costs no extra parameters. *logits*: one raw score per character per position, before softmax.

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, n_layer, n_embd, n_head, block_size, norm="pre"):
        super().__init__()
        self.block_size = block_size
        self.wte = nn.Embedding(vocab_size, n_embd)             # figure 1: Output Embedding
        self.wpe = nn.Embedding(block_size, n_embd)             # figure 1: Positional Encoding (learned here)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, block_size, norm) for _ in range(n_layer)])   # N x
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)   # figure 1: Linear
        self.lm_head.weight = self.wte.weight                    # weight tying, as in nanoGPT
        self.apply(lambda m: nn.init.normal_(m.weight, std=0.02) if isinstance(m, (nn.Linear, nn.Embedding)) else None)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.wte(idx) + self.wpe(torch.arange(T, device=idx.device))   # the residual stream starts here
        for blk in self.blocks:
            x = blk(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

In [ ]:
#@title 🔮 Predict C1: Total parameters for this config? Lesson 3's formula: 12·d²·N + V·d + T·d. Write your number (in millions or thousands).
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("C1", prediction, confidence)

In [ ]:
model = GPT(vocab_size, cfg["n_layer"], cfg["n_embd"], cfg["n_head"], cfg["block_size"]).to(device)
n_total = sum(p.numel() for p in model.parameters())          # tied weights are counted once
d, N, T = cfg["n_embd"], cfg["n_layer"], cfg["block_size"]
formula = 12 * d * d * N + vocab_size * d + T * d
print(f"counted {n_total:,}   formula {formula:,}   gap {n_total - formula:,} (biases and LayerNorm gains)")
for name, m in model.named_children():
    k = sum(p.numel() for p in m.parameters())
    print(f"  {name:8s} {k:>12,}" + ("   (same tensor as wte)" if name == "lm_head" else ""))

In [ ]:
#@title ✍️ C1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("C1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict C2: Loss of the UNTRAINED model on one batch? Hint: if all V characters were equally likely, cross-entropy would be ln(V).
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("C2", prediction, confidence)

In [ ]:
xb, yb = get_batch("train")
logits, loss = model(xb, yb)
print("logits", tuple(logits.shape))
print(f"loss at init {loss.item():.3f}   ln(vocab_size) = {math.log(vocab_size):.3f}")

Surprise? With small initial weights the head is nearly uniform, so the starting loss sits at ln V: the model is guessing. Everything training does is measured as distance below that line.

---
## Part D · Train (50 min · T4 recommended)

**Words used in this part.** *loss curve*: loss against step; read its slope, not one value. *validation loss*: the loss on the 10% never trained on; the number you report. *checkpoint*: the weights saved to disk. *run log*: one line per run with config, seed and final losses, so a result can be repeated.

On the T4 the real run (`SMOKE = False`, 3000 steps) takes about 15 minutes. On CPU it would take hours: keep `SMOKE = True` there.

In [ ]:
@torch.no_grad()
def estimate_loss(model, iters=10):
    model.eval(); out = {}
    for split in ("train", "val"):
        out[split] = torch.stack([model(*get_batch(split))[1] for _ in range(iters)]).mean().item()
    model.train(); return out

def train(model, steps, lr, eval_every, tag=""):
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    log, t0 = [], time.time()
    for step in range(steps + 1):
        if step % eval_every == 0 or step == steps:
            l = estimate_loss(model); log.append((step, l["train"], l["val"]))
            print(f"{tag:9s} step {step:5d}  train {l['train']:.3f}  val {l['val']:.3f}  {time.time() - t0:5.0f}s")
        if step == steps:
            break
        xb, yb = get_batch("train")
        _, loss = model(xb, yb)                                   # forward
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()   # backward, update
    return log

In [ ]:
#@title 🔮 Predict D1: After cfg['steps'] steps, roughly what VALIDATION loss? It starts near ln V (about 4.2); nanoGPT's full run reaches about 1.5.
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D1", prediction, confidence)

In [ ]:
torch.manual_seed(cfg["seed"])
model = GPT(vocab_size, cfg["n_layer"], cfg["n_embd"], cfg["n_head"], cfg["block_size"]).to(device)
log = train(model, cfg["steps"], cfg["lr"], cfg["eval_every"], tag="pre-norm")
steps_, tr, va = zip(*log)
plt.plot(steps_, tr, label="train"); plt.plot(steps_, va, label="val")
plt.axhline(math.log(vocab_size), ls="--", c="grey", label="ln(vocab): guessing")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.show()

In [ ]:
#@title ✍️ D1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D1", what_happened, surprise)

**Reading the curve.** The fast early drop is character frequencies (spaces, `e`, `t`). The slow middle is spelling and names. If the train line keeps falling while val flattens or rises, the model is memorising the training text; more data or fewer steps, not more model.

In [ ]:
torch.save(model.state_dict(), "week05_ckpt.pt")            # the checkpoint: every weight, ~4 bytes each
run = dict(config=cfg, vocab_size=vocab_size, smoke=SMOKE, device=device,
           final_train=round(log[-1][1], 4), final_val=round(log[-1][2], 4), when=time.strftime("%Y-%m-%d %H:%M"))
with open("week05_runs.jsonl", "a") as f:                    # one line per run: this is your run log
    f.write(json.dumps(run) + "\n")
print(f"checkpoint {os.path.getsize('week05_ckpt.pt') / 1e6:.2f} MB for {n_total:,} weights")
print(json.dumps(run, indent=1))
print("Colab deletes these files when the session ends: download both from the Files panel.")

---
## Part E · Sampling (25 min)

**Words used in this part.** *sampling*: draw one character in proportion to the model's probabilities, append it, repeat. *temperature*: divide the logits by T before softmax; below 1 sharpens, above 1 flattens. *top-k*: keep only the k most likely characters. *greedy decoding*: always the most likely character (T → 0).

In [ ]:
@torch.no_grad()
def generate(model, prompt, n=200, temperature=1.0, top_k=None):
    model.eval()
    idx = torch.tensor([encode(prompt)], device=device)
    for _ in range(n):
        idx_cond = idx[:, -model.block_size:]                    # crop to the context length
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature                  # nanoGPT: the temperature line
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("Inf")          # nanoGPT: the top-k line
        probs = F.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, num_samples=1)            # one draw
        idx = torch.cat((idx, nxt), dim=1)
    model.train()
    return decode(idx[0].tolist())

In [ ]:
#@title 🔮 Predict E1: Same prompt at temperature 0.01, 0.3, 1.0 and 2.0: which one repeats itself, which one is spelling noise?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E1", prediction, confidence)

In [ ]:
for T in [0.01, 0.3, 1.0, 2.0]:
    torch.manual_seed(cfg["seed"])
    print(f"--- temperature {T} ---")
    print(generate(model, "\n", n=150, temperature=T))

In [ ]:
#@title ✍️ E1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("E1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict E2: Two samples with the SAME seed and settings: identical or not? A third with a different seed? And does top-k = 5 tame temperature 2.0?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E2", prediction, confidence)

In [ ]:
torch.manual_seed(cfg["seed"]); a = generate(model, "\n", n=120, temperature=1.0, top_k=5)
torch.manual_seed(cfg["seed"]); b = generate(model, "\n", n=120, temperature=1.0, top_k=5)
torch.manual_seed(cfg["seed"] + 1); c = generate(model, "\n", n=120, temperature=1.0, top_k=5)
print("same seed -> identical:", a == b, "| different seed -> identical:", a == c)
print(a)
torch.manual_seed(cfg["seed"])
print("--- temperature 2.0 with top_k=5 ---")
print(generate(model, "\n", n=150, temperature=2.0, top_k=5))

Surprise? A seed plus the two knobs reproduces a sample exactly; change any one and it is a different stimulus. That triple belongs in every run log and every method section.

---
## Part F · Figure 1 → code, then break the block (30 min)

**Words used in this part.** *pre-norm / post-norm*: LayerNorm on the branch (`x + f(LN(x))`) versus on the sum (`LN(x + f(x))`). *depth*: the number of blocks, `n_layer`.

Every box of the paper's figure 1, mapped to a name in this notebook:

| Figure 1 box | In this notebook |
|---|---|
| Output Embedding | `GPT.wte` |
| Positional Encoding | `GPT.wpe`, added in `GPT.forward` |
| Masked Multi-Head Attention | `CausalSelfAttention` (the `mask` buffer is "masked") |
| Add & Norm | `x = x + ...` and `ln_1` / `ln_2` in `Block.forward` (pre-norm here) |
| Feed Forward | `MLP` |
| N× | `GPT.blocks`, `n_layer` of them |
| Linear | `GPT.lm_head` (tied to `wte`) |
| Softmax | `F.softmax` in `generate`; hidden inside `cross_entropy` in training |
| Left stack, cross-attention | not in a GPT |

In [ ]:
#@title 🔮 Predict F1: Same seed, same steps, post-norm instead of pre-norm: which reaches the lower validation loss, and by roughly how much?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("F1", prediction, confidence)

In [ ]:
results = {}
for norm in ["pre", "post"]:
    torch.manual_seed(cfg["seed"])
    m = GPT(vocab_size, cfg["n_layer"], cfg["n_embd"], cfg["n_head"], cfg["block_size"], norm=norm).to(device)
    lg = train(m, cfg["steps"] // 2, cfg["lr"], cfg["eval_every"], tag=norm + "-norm")
    results[norm] = lg[-1][2]
print("final val loss:", {k: round(v, 3) for k, v in results.items()})

In [ ]:
#@title ✍️ F1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("F1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict F2: Stretch: 1, 2 and 4 blocks at the same width and steps. Does validation loss improve with each doubling, and by how much?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("F2", prediction, confidence)

In [ ]:
RUN_STRETCH = True  #@param {type:"boolean"}
if RUN_STRETCH:
    for n_layer in [1, 2, 4]:
        torch.manual_seed(cfg["seed"])
        m = GPT(vocab_size, n_layer, cfg["n_embd"], cfg["n_head"], cfg["block_size"]).to(device)
        k = cfg["steps"] // 3
        lg = train(m, k, cfg["lr"], max(cfg["eval_every"], k), tag=f"{n_layer} blocks")
        print(f"   {n_layer} blocks, {sum(p.numel() for p in m.parameters()):,} params -> val {lg[-1][2]:.3f}")

Surprise? In a short run more depth can *lose*: each block adds parameters that need steps to earn their keep. Depth pays at the number of steps and data of the real run, not at a few hundred. That is the honest version of "bigger is better".

---
## Wrap up

Run the cell below, copy the table into your insight ledger next to `core-transformer`, and mark the biggest surprise. Then on the laptop:

```
python track.py done week-05/build --rating <1-5> --minutes 180 --note "biggest surprise"
python gen_week.py 6
```

In [ ]:
reveal()